In [2]:
from pathlib import Path

import pandas as pd

In [3]:
PROJECT_ROOT = Path.cwd().parent

HISTORICAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "historical"
    / "fipe_history_2026_08.parquet"
)

HISTORICAL_PATH

WindowsPath('e:/VSCODE Files/Projects/02_fipe_data_pipeline/data/raw/historical/fipe_history_2026_08.parquet')

In [186]:
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 20)

In [4]:
df_history = pd.read_parquet(HISTORICAL_PATH)

In [12]:
df_history.shape

(9478205, 12)

In [13]:
df_history.head()

,tipo_veiculo,codigo_fipe,nome_modelo,nome_marca,nome_combustivel,sigla_combustivel,ano_modelo,zero_km,valor_centavos,valor_formatado,mes_referencia,ano_referencia
0,moto,840015-6,ATV 100,ADLY,Gasolina,g,1990.0,False,0,"R$ 0,00",9,2012
1,moto,840015-6,ATV 100,ADLY,Gasolina,g,1991.0,False,0,"R$ 0,00",9,2012
2,moto,840015-6,ATV 100,ADLY,Gasolina,g,1992.0,False,0,"R$ 0,00",9,2012
3,moto,840015-6,ATV 100,ADLY,Gasolina,g,1993.0,False,0,"R$ 0,00",9,2012
4,moto,840015-6,ATV 100,ADLY,Gasolina,g,1994.0,False,0,"R$ 0,00",9,2012


In [14]:
df_history.info()

<class 'pandas.DataFrame'>
RangeIndex: 9478205 entries, 0 to 9478204
Data columns (total 12 columns):
 #   Column             Dtype  
---  ------             -----  
 0   tipo_veiculo       str    
 1   codigo_fipe        str    
 2   nome_modelo        str    
 3   nome_marca         str    
 4   nome_combustivel   str    
 5   sigla_combustivel  str    
 6   ano_modelo         float64
 7   zero_km            bool   
 8   valor_centavos     int64  
 9   valor_formatado    str    
 10  mes_referencia     int32  
 11  ano_referencia     int32  
dtypes: bool(1), float64(1), int32(2), int64(1), str(7)
memory usage: 1.3 GB


In [15]:
df_history.columns.tolist()

['tipo_veiculo',
 'codigo_fipe',
 'nome_modelo',
 'nome_marca',
 'nome_combustivel',
 'sigla_combustivel',
 'ano_modelo',
 'zero_km',
 'valor_centavos',
 'valor_formatado',
 'mes_referencia',
 'ano_referencia']

In [16]:
df_history[
    ["ano_referencia", "mes_referencia"]
].drop_duplicates().sort_values(
    ["ano_referencia", "mes_referencia"]
)

,ano_referencia,mes_referencia
967,2001,1
966,2001,2
965,2001,3
964,2001,4
313,2001,5
...,...,...
14,2026,4
13,2026,5
12,2026,6
11,2026,7


In [17]:
df_history["ano_referencia"].min(), df_history["ano_referencia"].max()

(np.int32(2001), np.int32(2026))

In [18]:
df_history["mes_referencia"].unique()

array([ 9,  8,  7,  6,  5,  4,  3,  2,  1, 12, 11, 10], dtype=int32)

In [19]:
df_history[
    ["ano_referencia", "mes_referencia"]
].drop_duplicates().sort_values(
    ["ano_referencia", "mes_referencia"],
    ascending=[False, False]
).head(12)

,ano_referencia,mes_referencia
10,2026,8
11,2026,7
12,2026,6
13,2026,5
14,2026,4
15,2026,3
16,2026,2
17,2026,1
18,2025,12
19,2025,11


In [20]:
df_history[
    (df_history["ano_referencia"] == 2026)
    & (df_history["mes_referencia"] == 8)
].shape

(50838, 12)

In [49]:
df_history[
    df_history["ano_referencia"] == 2026
].groupby("mes_referencia").size()

mes_referencia
1    49676
2    49773
3    49987
4    50129
5    50252
6    50395
7    50599
8    50838
dtype: int64

In [22]:
grain_columns = [
    "ano_referencia",
    "mes_referencia",
    "codigo_fipe",
    "ano_modelo",
    "sigla_combustivel",
]

In [ ]:
duplicates = df_history.duplicated(
    subset=grain_columns,
    keep=False
)

duplicates.sum()

np.int64(324)

In [58]:
duplicates

0          False
1          False
2          False
3          False
4          False
           ...  
9478200    False
9478201    False
9478202    False
9478203    False
9478204    False
Length: 9478205, dtype: bool

In [24]:
df_history.loc[
    duplicates,
    grain_columns
].sort_values(grain_columns).head(20)

,ano_referencia,mes_referencia,codigo_fipe,ano_modelo,sigla_combustivel
788816,2021,1,040001-7,1988.0,g
799538,2021,1,040001-7,1988.0,g
788916,2021,1,040001-7,1989.0,g
799747,2021,1,040001-7,1989.0,g
789016,2021,1,040001-7,1990.0,g
799956,2021,1,040001-7,1990.0,g
789116,2021,1,040001-7,1991.0,g
800165,2021,1,040001-7,1991.0,g
789216,2021,1,040001-7,1992.0,g
800374,2021,1,040001-7,1992.0,g


In [25]:
len(df_history)

9478205

In [26]:
df_history[grain_columns].drop_duplicates().shape[0]

9478028

In [27]:
duplicate_groups = (
    df_history
    .groupby(grain_columns)
    .size()
    .reset_index(name="row_count")
    .query("row_count > 1")
    .sort_values("row_count", ascending=False)
)

duplicate_groups

,ano_referencia,mes_referencia,codigo_fipe,ano_modelo,sigla_combustivel,row_count
5983608,2021,1,825077-4,2021.0,g,3
5983607,2021,1,825077-4,2020.0,g,3
5985723,2021,1,872002-9,2011.0,g,3
5985724,2021,1,872002-9,2014.0,g,3
6024029,2021,2,825077-4,2020.0,g,3
...,...,...,...,...,...,...
7026950,2023,2,530029-0,2023.0,d,2
7026949,2023,2,530029-0,2022.0,d,2
7026954,2023,2,530031-2,2021.0,d,2
7026955,2023,2,530031-2,2022.0,d,2


In [28]:
duplicate_groups["row_count"].value_counts().sort_index()

row_count
2    101
3     27
Name: count, dtype: int64

In [29]:
sample_key = duplicate_groups.iloc[0]

sample_duplicates = df_history[
    (df_history["ano_referencia"] == sample_key["ano_referencia"])
    & (df_history["mes_referencia"] == sample_key["mes_referencia"])
    & (df_history["codigo_fipe"] == sample_key["codigo_fipe"])
    & (df_history["ano_modelo"] == sample_key["ano_modelo"])
    & (df_history["sigla_combustivel"] == sample_key["sigla_combustivel"])
]

sample_duplicates

,tipo_veiculo,codigo_fipe,nome_modelo,nome_marca,nome_combustivel,sigla_combustivel,ano_modelo,zero_km,valor_centavos,valor_formatado,mes_referencia,ano_referencia
7427418,moto,825077-4,DL 1000 V-STROM ADVENTURE,SUZUKI,Gasolina,g,2021.0,False,6023000,"R$ 60.230,00",1,2021
7427419,moto,825077-4,DL 1000 V-STROM ADVENTURE,SUZUKI,Gasolina,g,2021.0,False,6023000,"R$ 60.230,00",1,2021
7427420,moto,825077-4,DL 1000 V-STROM ADVENTURE,SUZUKI,Gasolina,g,2021.0,False,6023000,"R$ 60.230,00",1,2021


In [80]:
full_duplicates = df_history.duplicated(
    subset=df_history.columns.tolist(), # Não é necessário
    keep=False
)

full_duplicates.sum()

np.int64(156)

In [31]:
exact_duplicate_groups = (
    df_history.loc[full_duplicates]
    .groupby(df_history.columns.tolist(), dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
)

exact_duplicate_groups["row_count"].value_counts().sort_index()

row_count
2    63
3    10
Name: count, dtype: int64

In [32]:
len(df_history) - len(df_history.drop_duplicates())

83

In [116]:
duplicate_groups = (
    df_history
    .groupby(grain_columns, dropna=False)
    .size()
    .reset_index(name="row_count")
    .query("row_count > 1")
    .sort_values("row_count", ascending=False)
)

duplicate_groups["row_count"].value_counts().sort_index()

row_count
2    117
3     30
Name: count, dtype: int64

In [117]:
duplicate_groups["row_count"].sum()

np.int64(324)

In [35]:
(duplicate_groups["row_count"] - 1).sum()

np.int64(177)

In [36]:
grain_duplicates = df_history[
    df_history.duplicated(
        subset=grain_columns,
        keep=False
    )
].copy()

exact_duplicate_mask = grain_duplicates.duplicated(
    subset=df_history.columns.tolist(),
    keep=False
)

non_exact_duplicates = grain_duplicates[
    ~exact_duplicate_mask
].copy()

non_exact_duplicates.shape

(168, 12)

In [132]:
non_exact_duplicates.sort_values(
    grain_columns
).head(20)

,tipo_veiculo,codigo_fipe,nome_modelo,nome_marca,nome_combustivel,sigla_combustivel,ano_modelo,zero_km,valor_centavos,valor_formatado,mes_referencia,ano_referencia
788816,carro,040001-7,Buggy 1.6 2-Lug.,Baby,Gasolina,g,1988.0,False,221600,"R$ 2.216,00",1,2021
799538,carro,040001-7,Buggy 1.6 2-Lug.,Buggy,Gasolina,g,1988.0,False,221600,"R$ 2.216,00",1,2021
788916,carro,040001-7,Buggy 1.6 2-Lug.,Baby,Gasolina,g,1989.0,False,281400,"R$ 2.814,00",1,2021
799747,carro,040001-7,Buggy 1.6 2-Lug.,Buggy,Gasolina,g,1989.0,False,281400,"R$ 2.814,00",1,2021
789016,carro,040001-7,Buggy 1.6 2-Lug.,Baby,Gasolina,g,1990.0,False,322800,"R$ 3.228,00",1,2021
799956,carro,040001-7,Buggy 1.6 2-Lug.,Buggy,Gasolina,g,1990.0,False,322800,"R$ 3.228,00",1,2021
789116,carro,040001-7,Buggy 1.6 2-Lug.,Baby,Gasolina,g,1991.0,False,336300,"R$ 3.363,00",1,2021
800165,carro,040001-7,Buggy 1.6 2-Lug.,Buggy,Gasolina,g,1991.0,False,336300,"R$ 3.363,00",1,2021
789216,carro,040001-7,Buggy 1.6 2-Lug.,Baby,Gasolina,g,1992.0,False,346900,"R$ 3.469,00",1,2021
800374,carro,040001-7,Buggy 1.6 2-Lug.,Buggy,Gasolina,g,1992.0,False,346900,"R$ 3.469,00",1,2021


In [126]:
comparison = (
    non_exact_duplicates
    .groupby(grain_columns, dropna=False)
    .agg({
        col: "nunique"
        for col in df_history.columns
        if col not in grain_columns
    })
)

comparison = comparison[
    (comparison > 1).any(axis=1)
]

comparison.head(20)

tipo_veiculo  \
ano_referencia mes_referencia codigo_fipe ano_modelo sigla_combustivel                 
2021           1              040001-7    1988.0     g                             1   
                                          1989.0     g                             1   
                                          1990.0     g                             1   
                                          1991.0     g                             1   
                                          1992.0     g                             1   
                                          1993.0     g                             1   
                                          1994.0     g                             1   
                                          1995.0     g                             1   
                                          1996.0     g                             1   
                                          1997.0     g                             1   
                              832001-2    1997.0     g                             1   
                                          1998.0     g                             1   
                                          1999.0     g                             1   
                                          2000.0     g                             1   
                                          2001.0     g                             1   
                                          2002.0     g                             1   
                              832002-0    1997.0     g                             1   
                                          1998.0     g                             1   
                                          1999.0     g                             1   
                                          2000.0     g                             1   

                                                                        nome_modelo  \
ano_referencia mes_referencia codigo_fipe ano_modelo sigla_combustivel                
2021           1              040001-7    1988.0     g                            1   
                                          1989.0     g                            1   
                                          1990.0     g                            1   
                                          1991.0     g                            1   
                                          1992.0     g                            1   
                                          1993.0     g                            1   
                                          1994.0     g                            1   
                                          1995.0     g                            1   
                                          1996.0     g                            1   
                                          1997.0     g                            1   
                              832001-2    1997.0     g                            1   
                                          1998.0     g                            1   
                                          1999.0     g                            1   
                                          2000.0     g                            1   
                                          2001.0     g                            1   
                                          2002.0     g                            1   
                              832002-0    1997.0     g                            1   
                                          1998.0     g                            1   
                                          1999.0     g                            1   
                                          2000.0     g                            1   

                                                                        nome_marca  \
ano_referencia mes_referencia codigo_fipe ano_modelo sigla_combustivel               
2021           1              040001-7    1988.0

In [40]:
variation_summary = (
    comparison.gt(1)
    .sum()
    .sort_values(ascending=False)
)

variation_summary

nome_marca          68
nome_modelo          6
tipo_veiculo         0
nome_combustivel     0
zero_km              0
valor_centavos       0
valor_formatado      0
dtype: int64

In [134]:
model_conflicts = comparison[
    comparison["nome_modelo"] > 1
].reset_index()

model_conflicts

,ano_referencia,mes_referencia,codigo_fipe,ano_modelo,sigla_combustivel,tipo_veiculo,nome_modelo,nome_marca,nome_combustivel,zero_km,valor_centavos,valor_formatado
0,2021,11,033182-1,2021.0,h,1,2,1,1,1,1,1
1,2021,11,033182-1,2022.0,h,1,2,1,1,1,1,1
2,2021,11,033182-1,NaN,h,1,2,1,1,1,1,1
3,2021,11,087007-2,2021.0,g,1,2,1,1,1,1,1
4,2021,11,087007-2,2022.0,g,1,2,1,1,1,1,1
5,2021,11,087007-2,NaN,g,1,2,1,1,1,1,1


In [135]:
model_conflict_rows = df_history.merge(
    model_conflicts[grain_columns],
    on=grain_columns,
    how="inner"
)

model_conflict_rows.sort_values(
    grain_columns + ["nome_modelo"]
)

,tipo_veiculo,codigo_fipe,nome_modelo,nome_marca,nome_combustivel,sigla_combustivel,ano_modelo,zero_km,valor_centavos,valor_formatado,mes_referencia,ano_referencia
0,carro,033182-1,Range.R. SP.HSE Dyn.BL. 2.0 Si4(Hibrido),Land Rover,Híbrido,h,2021.0,False,61239800,"R$ 612.398,00",11,2021
3,carro,033182-1,Range.R. Sp.HSE Dyn.Bi. 2.0 Si4(Hibrido),Land Rover,Híbrido,h,2021.0,False,61239800,"R$ 612.398,00",11,2021
1,carro,033182-1,Range.R. SP.HSE Dyn.BL. 2.0 Si4(Hibrido),Land Rover,Híbrido,h,2022.0,False,64227000,"R$ 642.270,00",11,2021
4,carro,033182-1,Range.R. Sp.HSE Dyn.Bi. 2.0 Si4(Hibrido),Land Rover,Híbrido,h,2022.0,False,64227000,"R$ 642.270,00",11,2021
2,carro,033182-1,Range.R. SP.HSE Dyn.BL. 2.0 Si4(Hibrido),Land Rover,Híbrido,h,NaN,True,67882100,"R$ 678.821,00",11,2021
5,carro,033182-1,Range.R. Sp.HSE Dyn.Bi. 2.0 Si4(Hibrido),Land Rover,Híbrido,h,NaN,True,67882100,"R$ 678.821,00",11,2021
6,carro,087007-2,Cullinan Black Badge 6.7 V12 Aut.,Rolls-Royce,Gasolina,g,2021.0,False,598796300,"R$ 5.987.963,00",11,2021
9,carro,087007-2,CullinanBlack Badged 6.7 V12 Aut.,Rolls-Royce,Gasolina,g,2021.0,False,598796300,"R$ 5.987.963,00",11,2021
7,carro,087007-2,Cullinan Black Badge 6.7 V12 Aut.,Rolls-Royce,Gasolina,g,2022.0,False,620212000,"R$ 6.202.120,00",11,2021
10,carro,087007-2,CullinanBlack Badged 6.7 V12 Aut.,Rolls-Royce,Gasolina,g,2022.0,False,620212000,"R$ 6.202.120,00",11,2021


In [137]:
null_summary = (
    df_history
    .isna()
    .sum()
    .sort_values(ascending=False)
)

null_summary

ano_modelo           506004
tipo_veiculo              0
nome_modelo               0
codigo_fipe               0
nome_marca                0
nome_combustivel          0
sigla_combustivel         0
zero_km                   0
valor_centavos            0
valor_formatado           0
mes_referencia            0
ano_referencia            0
dtype: int64

In [138]:
null_percentage = (
    df_history
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

null_percentage

ano_modelo           5.338606
tipo_veiculo         0.000000
nome_modelo          0.000000
codigo_fipe          0.000000
nome_marca           0.000000
nome_combustivel     0.000000
sigla_combustivel    0.000000
zero_km              0.000000
valor_centavos       0.000000
valor_formatado      0.000000
mes_referencia       0.000000
ano_referencia       0.000000
dtype: float64

In [141]:
pd.crosstab(
    df_history["zero_km"],
    df_history["ano_modelo"].isna()
)

ano_modelo,False,True
zero_km,,
False,8972201,0
True,0,506004


In [142]:
invalid_zero_km = df_history[
    (df_history["zero_km"] == True)
    & (df_history["ano_modelo"].notna())
]

invalid_zero_km.shape

(0, 12)

In [143]:
invalid_used = df_history[
    (df_history["zero_km"] == False)
    & (df_history["ano_modelo"].isna())
]

invalid_used.shape

(0, 12)

In [144]:
structural_columns = [
    "tipo_veiculo",
    "codigo_fipe",
    "nome_modelo",
    "nome_marca",
    "nome_combustivel",
    "sigla_combustivel",
    "zero_km",
    "valor_centavos",
    "mes_referencia",
    "ano_referencia",
]

df_history[structural_columns].isna().sum()

tipo_veiculo         0
codigo_fipe          0
nome_modelo          0
nome_marca           0
nome_combustivel     0
sigla_combustivel    0
zero_km              0
valor_centavos       0
mes_referencia       0
ano_referencia       0
dtype: int64

In [146]:
df_history.dtypes

tipo_veiculo             str
codigo_fipe              str
nome_modelo              str
nome_marca               str
nome_combustivel         str
sigla_combustivel        str
ano_modelo           float64
zero_km                 bool
valor_centavos         int64
valor_formatado          str
mes_referencia         int32
ano_referencia         int32
dtype: object

In [147]:
for col in [
    "tipo_veiculo",
    "sigla_combustivel",
    "nome_combustivel",
    "zero_km",
    "mes_referencia",
    "ano_referencia",
]:
    print(f"\n### {col}")
    print(df_history[col].value_counts(dropna=False).sort_index())


### tipo_veiculo
tipo_veiculo
caminhão    2154948
carro       5831745
moto        1491512
Name: count, dtype: int64

### sigla_combustivel
sigla_combustivel
d    3095651
e     132772
f     956996
g    5206230
h      49101
l      32067
n       5388
Name: count, dtype: int64

### nome_combustivel
nome_combustivel
Diesel         3095651
Elétrico         32067
Flex            956996
Gasolina       5206230
Gás Natural       5388
Híbrido          49101
Álcool          132772
Name: count, dtype: int64

### zero_km
zero_km
False    8972201
True      506004
Name: count, dtype: int64

### mes_referencia
mes_referencia
1     790667
2     792303
3     794311
4     797907
5     801397
6     804390
7     807645
8     810752
9     763455
10    767543
11    772015
12    775820
Name: count, dtype: int64

### ano_referencia
ano_referencia
2001    151191
2002    172862
2003    185851
2004    197349
2005    211732
2006    228813
2007    245932
2008    262443
2009    279941
2010    299752
2011    320376
2

In [148]:
df_history["ano_modelo"].dtype

dtype('float64')

In [149]:
df_history["ano_modelo"].dropna().describe()

count    8.972201e+06
mean     2.003990e+03
std      9.467828e+00
min      1.981000e+03
25%      1.997000e+03
50%      2.003000e+03
75%      2.011000e+03
max      2.027000e+03
Name: ano_modelo, dtype: float64

In [150]:
df_history["ano_modelo"].dropna().sort_values().head(20)

5251466    1981.0
5251465    1981.0
5251464    1981.0
5251463    1981.0
5251462    1981.0
5251461    1981.0
5251428    1981.0
8089039    1981.0
8089038    1981.0
8089030    1981.0
8089058    1981.0
8089057    1981.0
8089034    1981.0
8089033    1981.0
8089032    1981.0
8089031    1981.0
5251477    1981.0
5251444    1981.0
5251459    1981.0
5251458    1981.0
Name: ano_modelo, dtype: float64

In [151]:
df_history["ano_modelo"].dropna().sort_values(ascending=False).head(20)

5865034    2027.0
5865035    2027.0
8254200    2027.0
6757040    2027.0
3218743    2027.0
3218744    2027.0
3218745    2027.0
8201790    2027.0
8201791    2027.0
5865036    2027.0
8201792    2027.0
6803940    2027.0
3107970    2027.0
8254201    2027.0
6803941    2027.0
8201793    2027.0
7310715    2027.0
1284051    2027.0
3451402    2027.0
1284052    2027.0
Name: ano_modelo, dtype: float64

In [152]:
fuel_mapping = (
    df_history[
        ["sigla_combustivel", "nome_combustivel"]
    ]
    .drop_duplicates()
    .sort_values("sigla_combustivel")
)

fuel_mapping

,sigla_combustivel,nome_combustivel
5029,d,Diesel
1723229,e,Álcool
226402,f,Flex
0,g,Gasolina
226683,h,Híbrido
112450,l,Elétrico
1765916,n,Gás Natural


In [153]:
df_history.groupby(
    "sigla_combustivel"
)["nome_combustivel"].nunique()

sigla_combustivel
d    1
e    1
f    1
g    1
h    1
l    1
n    1
Name: nome_combustivel, dtype: int64

In [189]:
model_year_gap = (
    df_history.loc[
        df_history["ano_modelo"].notna()
    ].assign(
        year_gap=lambda x:
        x["ano_modelo"] - x["ano_referencia"]
    )
)

model_year_gap["year_gap"].value_counts().sort_index()

year_gap
-45.0       720
-44.0      1856
-43.0      3036
-42.0      4336
-41.0      6812
-40.0      9984
-39.0     13368
-38.0     16856
-37.0     20664
-36.0     24640
-35.0     29586
-34.0     35882
-33.0     43711
-32.0     53311
-31.0     65269
-30.0     77412
-29.0     89865
-28.0    103883
-27.0    119025
-26.0    134204
-25.0    149373
-24.0    163045
-23.0    175417
-22.0    187369
-21.0    200117
-20.0    214075
-19.0    227286
-18.0    241377
-17.0    256155
-16.0    271568
-15.0    286660
-14.0    302870
-13.0    318304
-12.0    334079
-11.0    349839
-10.0    365257
-9.0     378019
-8.0     389283
-7.0     398622
-6.0     405777
-5.0     408550
-4.0     412157
-3.0     415709
-2.0     415238
-1.0     409253
 0.0     356866
 1.0      85516
Name: count, dtype: int64

In [155]:
model_year_gap["year_gap"].min(), model_year_gap["year_gap"].max()

(np.float64(-45.0), np.float64(1.0))

In [156]:
cardinality_by_fipe = (
    df_history
    .groupby("codigo_fipe")
    .agg(
        tipo_veiculo_nunique=("tipo_veiculo", "nunique"),
        nome_marca_nunique=("nome_marca", "nunique"),
        nome_modelo_nunique=("nome_modelo", "nunique"),
        nome_combustivel_nunique=("nome_combustivel", "nunique"),
        sigla_combustivel_nunique=("sigla_combustivel", "nunique"),
    )
)

cardinality_by_fipe.describe()

,tipo_veiculo_nunique,nome_marca_nunique,nome_modelo_nunique,nome_combustivel_nunique,sigla_combustivel_nunique
count,11359.0,11359.000000,11359.000000,11359.000000,11359.000000
mean,1.0,1.001233,1.047011,1.016375,1.016375
std,0.0,0.043994,0.245203,0.126917,0.126917
min,1.0,1.000000,1.000000,1.000000,1.000000
25%,1.0,1.000000,1.000000,1.000000,1.000000
50%,1.0,1.000000,1.000000,1.000000,1.000000
75%,1.0,1.000000,1.000000,1.000000,1.000000
max,1.0,3.000000,5.000000,2.000000,2.000000


In [158]:
(cardinality_by_fipe > 1).sum()

tipo_veiculo_nunique           0
nome_marca_nunique            10
nome_modelo_nunique          455
nome_combustivel_nunique     186
sigla_combustivel_nunique    186
dtype: int64

In [190]:
cardinality_conflicts = cardinality_by_fipe[
    (cardinality_by_fipe > 1).any(axis=1)
]

cardinality_conflicts

,tipo_veiculo_nunique,nome_marca_nunique,nome_modelo_nunique,nome_combustivel_nunique,sigla_combustivel_nunique
codigo_fipe,,,,,
001001-4,1,1,1,2,2
001004-9,1,1,1,2,2
001005-7,1,1,1,2,2
001007-3,1,1,1,2,2
001008-1,1,1,1,2,2
...,...,...,...,...,...
891004-9,1,1,3,1,1
891005-7,1,1,3,1,1
893001-5,1,1,2,1,1


In [163]:
brand_conflicts = cardinality_by_fipe[
    cardinality_by_fipe["nome_marca_nunique"] > 1
]

brand_conflicts

,tipo_veiculo_nunique,nome_marca_nunique,nome_modelo_nunique,nome_combustivel_nunique,sigla_combustivel_nunique
codigo_fipe,,,,,
040001-7,1,2,1,1,1
040002-5,1,2,1,1,1
832001-2,1,2,1,1,1
832002-0,1,2,1,1,1
832027-6,1,2,1,1,1
832028-4,1,2,1,1,1
872001-0,1,3,1,1,1
872002-9,1,3,1,1,1
872003-7,1,3,1,1,1


In [164]:
df_history[
    df_history["codigo_fipe"].isin(brand_conflicts.index)
][
    ["codigo_fipe", "nome_marca"]
].drop_duplicates().sort_values("codigo_fipe")

,codigo_fipe,nome_marca
788749,040001-7,Baby
799538,040001-7,Buggy
789749,040002-5,Baby
789849,040002-5,Buggy
4378457,832001-2,KIMCO
4442669,832001-2,KYMCO
4379658,832002-0,KIMCO
4443977,832002-0,KYMCO
4380856,832027-6,KIMCO
4444649,832027-6,KYMCO


In [168]:
fuel_conflicts = cardinality_by_fipe[
    cardinality_by_fipe["sigla_combustivel_nunique"] > 1
]

df_history[
    df_history["codigo_fipe"].isin(fuel_conflicts.index)
][
    [
        "codigo_fipe",
        "nome_modelo",
        "ano_modelo",
        "ano_referencia",
        "mes_referencia",
        "nome_combustivel",
        "sigla_combustivel",
    ]
].drop_duplicates().sort_values("codigo_fipe")

,codigo_fipe,nome_modelo,ano_modelo,ano_referencia,mes_referencia,nome_combustivel,sigla_combustivel
1740717,001001-4,Fiorino Furgão 1.5 mpi / i.e.,1992.0,2026,8,Gasolina,g
1740718,001001-4,Fiorino Furgão 1.5 mpi / i.e.,1992.0,2026,7,Gasolina,g
1740719,001001-4,Fiorino Furgão 1.5 mpi / i.e.,1992.0,2026,6,Gasolina,g
1740720,001001-4,Fiorino Furgão 1.5 mpi / i.e.,1992.0,2026,5,Gasolina,g
1740721,001001-4,Fiorino Furgão 1.5 mpi / i.e.,1992.0,2026,4,Gasolina,g
...,...,...,...,...,...,...,...
112508,532001-1,ONE Work (Elétrico),NaN,2025,12,Elétrico,l
112509,532001-1,ONE Work (Elétrico),NaN,2025,11,Elétrico,l
112510,532001-1,ONE Work (Elétrico),NaN,2025,10,Elétrico,l
112511,532001-1,ONE Work (Elétrico),NaN,2025,9,Elétrico,l


In [173]:
fuel_conflict_summary = (
    df_history[
        df_history["codigo_fipe"].isin(fuel_conflicts.index)
    ]
    .groupby([
        "codigo_fipe",
        "nome_combustivel",
        "sigla_combustivel",
    ])
    .agg(
        occurrences=("codigo_fipe", "size"),
        first_year=("ano_referencia", "min"),
        last_year=("ano_referencia", "max"),
    )
    .reset_index()
    .sort_values(
        ["codigo_fipe", "occurrences"],
        ascending=[True, False]
    )
)

fuel_conflict_summary

,codigo_fipe,nome_combustivel,sigla_combustivel,occurrences,first_year,last_year
0,001001-4,Gasolina,g,3558,2001,2026
1,001001-4,Álcool,e,2000,2001,2026
2,001004-9,Gasolina,g,955,2001,2026
3,001004-9,Álcool,e,925,2001,2026
4,001005-7,Gasolina,g,964,2001,2026
...,...,...,...,...,...,...
367,502014-0,Gasolina,g,52,2025,2026
368,504074-4,Diesel,d,590,2001,2025
369,504074-4,Gasolina,g,26,2025,2026
370,532001-1,Elétrico,l,59,2025,2026


In [171]:
fuel_conflict_summary.groupby(
    "codigo_fipe"
)["occurrences"].agg(["min", "max"])

,min,max
codigo_fipe,,
001001-4,2000,3558
001004-9,925,955
001005-7,925,964
001007-3,924,1232
001008-1,616,1232
...,...,...
088004-3,2,24
096010-1,2,25
502014-0,52,1180


In [172]:
fuel_conflict_summary[
    fuel_conflict_summary["occurrences"] <= 3
]

,codigo_fipe,nome_combustivel,sigla_combustivel,occurrences,first_year,last_year
45,001073-1,Álcool,e,1,2001,2001
83,002066-4,Álcool,e,1,2016,2016
151,004018-5,Álcool,e,1,2002,2002
187,004091-6,Álcool,e,1,2003,2003
349,008315-1,Gasolina,g,2,2025,2025
357,071062-8,Gasolina,g,2,2025,2025
359,071063-6,Gasolina,g,2,2025,2025
361,088003-5,Gasolina,g,2,2025,2025
363,088004-3,Gasolina,g,2,2025,2025
364,096010-1,Gasolina,g,2,2025,2025


In [174]:
rare_fuel_codes = (
    fuel_conflict_summary[
        fuel_conflict_summary["occurrences"] <= 5
    ]["codigo_fipe"]
    .unique()
)

rare_fuel_history = (
    df_history[
        df_history["codigo_fipe"].isin(rare_fuel_codes)
    ][
        [
            "codigo_fipe",
            "nome_modelo",
            "ano_modelo",
            "ano_referencia",
            "mes_referencia",
            "nome_combustivel",
            "sigla_combustivel",
        ]
    ]
    .sort_values(
        [
            "codigo_fipe",
            "ano_referencia",
            "mes_referencia",
            "ano_modelo",
        ]
    )
)

rare_fuel_history

,codigo_fipe,nome_modelo,ano_modelo,ano_referencia,mes_referencia,nome_combustivel,sigla_combustivel
1908341,001073-1,Palio Weekend ELX 1.6 mpi,1998.0,2001,1,Gasolina,g
1908649,001073-1,Palio Weekend ELX 1.6 mpi,1999.0,2001,1,Gasolina,g
1908957,001073-1,Palio Weekend ELX 1.6 mpi,2000.0,2001,1,Gasolina,g
1908965,001073-1,Palio Weekend ELX 1.6 mpi,NaN,2001,1,Álcool,e
1908966,001073-1,Palio Weekend ELX 1.6 mpi,NaN,2001,1,Gasolina,g
...,...,...,...,...,...,...,...
112450,532001-1,ONE Work (Elétrico),2023.0,2026,8,Elétrico,l
112463,532001-1,ONE Work (Elétrico),2024.0,2026,8,Elétrico,l
112476,532001-1,ONE Work (Elétrico),2025.0,2026,8,Elétrico,l
112489,532001-1,ONE Work (Elétrico),2026.0,2026,8,Elétrico,l


In [175]:
rare_fuel_combinations = fuel_conflict_summary[
    fuel_conflict_summary["occurrences"] <= 5
][
    [
        "codigo_fipe",
        "nome_combustivel",
        "sigla_combustivel",
    ]
]

rare_fuel_rows = df_history.merge(
    rare_fuel_combinations,
    on=[
        "codigo_fipe",
        "nome_combustivel",
        "sigla_combustivel",
    ],
    how="inner"
)

rare_fuel_rows[
    [
        "codigo_fipe",
        "nome_modelo",
        "ano_modelo",
        "zero_km",
        "ano_referencia",
        "mes_referencia",
        "nome_combustivel",
        "sigla_combustivel",
    ]
].sort_values(
    [
        "codigo_fipe",
        "ano_referencia",
        "mes_referencia",
        "ano_modelo",
    ]
)

,codigo_fipe,nome_modelo,ano_modelo,zero_km,ano_referencia,mes_referencia,nome_combustivel,sigla_combustivel
6,001073-1,Palio Weekend ELX 1.6 mpi,NaN,True,2001,1,Álcool,e
19,002066-4,RAV4 2.0 4x4 16V Aut.,2010.0,False,2016,10,Álcool,e
12,004018-5,S10 Blazer Std. 2.2 MPFI / EFI,2000.0,False,2002,11,Álcool,e
11,004091-6,S10 Blazer Executive 4.3 V6,2001.0,False,2003,7,Álcool,e
4,008315-1,A6 Sport. e-tron Perf. Black (Elétrico),2025.0,False,2025,8,Gasolina,g
5,008315-1,A6 Sport. e-tron Perf. Black (Elétrico),NaN,True,2025,8,Gasolina,g
15,071062-8,Aceman E (Elétrico),2025.0,False,2025,8,Gasolina,g
16,071062-8,Aceman E (Elétrico),NaN,True,2025,8,Gasolina,g
17,071063-6,Aceman SE (Elétrico),2025.0,False,2025,8,Gasolina,g
18,071063-6,Aceman SE (Elétrico),NaN,True,2025,8,Gasolina,g


In [176]:
model_conflicts = cardinality_by_fipe[
    cardinality_by_fipe["nome_modelo_nunique"] > 1
]

model_conflicts.shape

(455, 5)

In [185]:
model_name_summary = (
    df_history[
        df_history["codigo_fipe"].isin(model_conflicts.index)
    ]
    .groupby(
        ["codigo_fipe", "nome_modelo"],
        dropna=False
    )
    .agg(
        occurrences=("nome_modelo", "size"),
        first_year=("ano_referencia", "min"),
        last_year=("ano_referencia", "max"),
    )
    .reset_index()
    .sort_values(
        ["codigo_fipe", "occurrences"],
        ascending=[True, False]
    )
)

model_name_summary

,codigo_fipe,nome_modelo,occurrences,first_year,last_year
1,001299-8,Stilo Duologic 1.8 ATTRACTIVE Flex 8V 5p,337,2009,2022
0,001299-8,Stilo Dualogic 1.8 ATTRACTIVE Flex 8V 5p,96,2022,2026
3,001487-7,Toro Freedom 2.0 16V 4x4 TB Diesel Aut.,317,2022,2026
2,001487-7,Toro Freedom 2.0 16V 4x4 Diesel Aut.,256,2017,2022
5,001509-1,ARGO 1.0 6V Flex.,480,2018,2025
...,...,...,...,...,...
984,893002-3,NQI Sport 1500W (Elétrica),184,2023,2026
986,893002-3,NQI Sport 1800W (Elétrica),8,2022,2023
985,893002-3,NQI Sport 1800W (Elétrica),2,2022,2022
987,893003-1,NQI GTS 3000W (Elétrica),184,2023,2026


In [178]:
model_conflicts["nome_modelo_nunique"].value_counts().sort_index()

nome_modelo_nunique
2    382
3     69
4      2
5      2
Name: count, dtype: int64

In [187]:
model_name_summary.groupby(
    "codigo_fipe"
)["occurrences"].agg(["min", "max"])

,min,max
codigo_fipe,,
001299-8,96,337
001487-7,256,317
001509-1,160,480
001516-4,172,338
001523-7,47,120
...,...,...
891004-9,29,108
891005-7,24,108
893001-5,10,184


In [180]:
model_month_conflicts = (
    df_history
    .groupby(
        [
            "codigo_fipe",
            "ano_referencia",
            "mes_referencia",
        ]
    )["nome_modelo"]
    .nunique()
)

model_month_conflicts = model_month_conflicts[
    model_month_conflicts > 1
]

model_month_conflicts

codigo_fipe  ano_referencia  mes_referencia
033182-1     2021            11                2
087007-2     2021            11                2
Name: nome_modelo, dtype: int64

In [181]:
model_month_conflicts.shape

(2,)

In [191]:
valor_formatado_numeric = (
    df_history["valor_formatado"]
    .str.replace("R$", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)

In [192]:
valor_centavos_numeric = (
    df_history["valor_centavos"] / 100
)

In [193]:
price_mismatch = (
    valor_formatado_numeric != valor_centavos_numeric
)

price_mismatch.sum()

np.int64(0)

In [194]:
df_history.loc[
    price_mismatch,
    [
        "codigo_fipe",
        "valor_centavos",
        "valor_formatado",
        "ano_referencia",
        "mes_referencia",
    ]
].head(50)

,codigo_fipe,valor_centavos,valor_formatado,ano_referencia,mes_referencia


In [195]:
monthly_volume = (
    df_history
    .groupby(
        ["ano_referencia", "mes_referencia"]
    )
    .size()
    .reset_index(name="rows")
    .sort_values(
        ["ano_referencia", "mes_referencia"]
    )
)

monthly_volume

,ano_referencia,mes_referencia,rows
0,2001,1,11844
1,2001,2,11660
2,2001,3,11107
3,2001,4,12171
4,2001,5,12566
...,...,...,...
303,2026,4,50129
304,2026,5,50252
305,2026,6,50395
306,2026,7,50599


In [196]:
monthly_volume["rows"].describe()

count      308.000000
mean     30773.392857
std      11411.734800
min      11107.000000
25%      20306.250000
50%      30821.500000
75%      40930.000000
max      50838.000000
Name: rows, dtype: float64

In [197]:
monthly_volume["reference_date"] = pd.to_datetime(
    dict(
        year=monthly_volume["ano_referencia"],
        month=monthly_volume["mes_referencia"],
        day=1,
    )
)

expected_months = pd.date_range(
    start=monthly_volume["reference_date"].min(),
    end=monthly_volume["reference_date"].max(),
    freq="MS",
)

missing_months = expected_months.difference(
    monthly_volume["reference_date"]
)

missing_months

DatetimeIndex([], dtype='datetime64[us]', freq='MS')

In [198]:
monthly_volume["pct_change"] = (
    monthly_volume["rows"]
    .pct_change()
    .mul(100)
)

monthly_volume[
    ["ano_referencia", "mes_referencia", "rows", "pct_change"]
].sort_values(
    "pct_change"
).head(20)

,ano_referencia,mes_referencia,rows,pct_change
2,2001,3,11107,-4.742710
27,2003,4,15231,-1.729144
1,2001,2,11660,-1.553529
157,2014,2,30880,-1.098549
85,2008,2,21272,-0.630635
110,2010,3,24221,-0.509345
197,2017,6,36565,-0.497986
76,2007,5,20161,-0.439506
36,2004,1,15985,-0.404984
176,2015,9,33870,-0.288507


In [199]:
monthly_volume[
    ["ano_referencia", "mes_referencia", "rows", "pct_change"]
].sort_values(
    "pct_change",
    ascending=False
).head(20)

,ano_referencia,mes_referencia,rows,pct_change
3,2001,4,12171,9.579544
4,2001,5,12566,3.245419
9,2001,10,13289,1.628939
13,2002,2,13938,1.611139
35,2003,12,16050,1.608002
111,2010,4,24562,1.407869
21,2002,10,14810,1.375864
118,2010,11,25770,1.301152
23,2002,12,15116,1.286518
44,2004,9,16640,1.247338


In [200]:
monthly_volume["pct_change"].describe()

count    307.000000
mean       0.478257
std        0.731587
min       -4.742710
25%        0.258190
50%        0.436500
75%        0.646017
max        9.579544
Name: pct_change, dtype: float64

In [201]:
df_history["codigo_fipe"].str.len().value_counts().sort_index()

codigo_fipe
8    9478205
Name: count, dtype: int64

In [202]:
valid_fipe_pattern = (
    df_history["codigo_fipe"]
    .str.fullmatch(r"\d{6}-\d")
)

valid_fipe_pattern.value_counts(dropna=False)

codigo_fipe
True    9478205
Name: count, dtype: int64

In [203]:
invalid_fipe_codes = df_history[
    ~valid_fipe_pattern
]

invalid_fipe_codes.shape

(0, 12)

In [204]:
invalid_fipe_codes[
    ["codigo_fipe", "nome_marca", "nome_modelo"]
].drop_duplicates().head(50)

,codigo_fipe,nome_marca,nome_modelo


In [205]:
(
    df_history["codigo_fipe"]
    != df_history["codigo_fipe"].str.strip()
).sum()

np.int64(0)

In [206]:
(
    df_history
    .groupby("codigo_fipe")["tipo_veiculo"]
    .nunique()
    .value_counts()
)

tipo_veiculo
1    11359
Name: count, dtype: int64

In [207]:
df_history["valor_centavos"].describe()

count    9.478205e+06
mean     9.170274e+06
std      2.173022e+07
min      0.000000e+00
25%      1.759000e+06
50%      3.841900e+06
75%      8.902000e+06
max      9.765744e+08
Name: valor_centavos, dtype: float64

In [208]:
(df_history["valor_centavos"] <= 0).sum()

np.int64(22)

In [209]:
df_history.nsmallest(
    20,
    "valor_centavos"
)[
    [
        "codigo_fipe",
        "nome_marca",
        "nome_modelo",
        "ano_modelo",
        "valor_centavos",
        "ano_referencia",
        "mes_referencia",
    ]
]

,codigo_fipe,nome_marca,nome_modelo,ano_modelo,valor_centavos,ano_referencia,mes_referencia
0,840015-6,ADLY,ATV 100,1990.0,0,2012,9
1,840015-6,ADLY,ATV 100,1991.0,0,2012,9
2,840015-6,ADLY,ATV 100,1992.0,0,2012,9
3,840015-6,ADLY,ATV 100,1993.0,0,2012,9
4,840015-6,ADLY,ATV 100,1994.0,0,2012,9
5,840015-6,ADLY,ATV 100,1995.0,0,2012,9
6,840015-6,ADLY,ATV 100,1996.0,0,2012,9
7,840015-6,ADLY,ATV 100,1997.0,0,2012,9
8,840015-6,ADLY,ATV 100,1998.0,0,2012,9
9,840015-6,ADLY,ATV 100,1999.0,0,2012,9


In [210]:
df_history.nlargest(
    20,
    "valor_centavos"
)[
    [
        "codigo_fipe",
        "nome_marca",
        "nome_modelo",
        "ano_modelo",
        "valor_centavos",
        "ano_referencia",
        "mes_referencia",
    ]
]

,codigo_fipe,nome_marca,nome_modelo,ano_modelo,valor_centavos,ano_referencia,mes_referencia
4557036,078017-0,LAMBORGHINI,AVENTADOR LP 770-4 SVJ,2022.0,976574400,2026,8
4557037,078017-0,LAMBORGHINI,AVENTADOR LP 770-4 SVJ,2022.0,976574400,2026,7
4557038,078017-0,LAMBORGHINI,AVENTADOR LP 770-4 SVJ,2022.0,976574400,2026,6
4557039,078017-0,LAMBORGHINI,AVENTADOR LP 770-4 SVJ,2022.0,969534300,2026,5
4557040,078017-0,LAMBORGHINI,AVENTADOR LP 770-4 SVJ,2022.0,960300000,2026,4
4557041,078017-0,LAMBORGHINI,AVENTADOR LP 770-4 SVJ,2022.0,960300000,2026,3
4556969,078017-0,LAMBORGHINI,AVENTADOR LP 770-4 SVJ,2021.0,952755500,2026,8
4556970,078017-0,LAMBORGHINI,AVENTADOR LP 770-4 SVJ,2021.0,952755500,2026,7
4556971,078017-0,LAMBORGHINI,AVENTADOR LP 770-4 SVJ,2021.0,952755500,2026,6
4556972,078017-0,LAMBORGHINI,AVENTADOR LP 770-4 SVJ,2021.0,945887100,2026,5


In [214]:
zero_price_rows = df_history[
    df_history["valor_centavos"] == 0
]

zero_price_rows[
    [
        "codigo_fipe",
        "nome_marca",
        "nome_modelo",
        "ano_modelo",
        "zero_km",
        "ano_referencia",
        "mes_referencia",
        "valor_centavos",
        "valor_formatado",
    ]
].sort_values(
    ["codigo_fipe", "ano_modelo"]
)

,codigo_fipe,nome_marca,nome_modelo,ano_modelo,zero_km,ano_referencia,mes_referencia,valor_centavos,valor_formatado
0,840015-6,ADLY,ATV 100,1990.0,False,2012,9,0,"R$ 0,00"
1,840015-6,ADLY,ATV 100,1991.0,False,2012,9,0,"R$ 0,00"
2,840015-6,ADLY,ATV 100,1992.0,False,2012,9,0,"R$ 0,00"
3,840015-6,ADLY,ATV 100,1993.0,False,2012,9,0,"R$ 0,00"
4,840015-6,ADLY,ATV 100,1994.0,False,2012,9,0,"R$ 0,00"
5,840015-6,ADLY,ATV 100,1995.0,False,2012,9,0,"R$ 0,00"
6,840015-6,ADLY,ATV 100,1996.0,False,2012,9,0,"R$ 0,00"
7,840015-6,ADLY,ATV 100,1997.0,False,2012,9,0,"R$ 0,00"
8,840015-6,ADLY,ATV 100,1998.0,False,2012,9,0,"R$ 0,00"
9,840015-6,ADLY,ATV 100,1999.0,False,2012,9,0,"R$ 0,00"


In [215]:
zero_price_rows[
    ["codigo_fipe", "ano_referencia", "mes_referencia"]
].value_counts()

codigo_fipe  ano_referencia  mes_referencia
840015-6     2012            9                 22
Name: count, dtype: int64

In [216]:
df_history[
    df_history["codigo_fipe"] == "840015-6"
][
    [
        "ano_modelo",
        "zero_km",
        "valor_centavos",
        "ano_referencia",
        "mes_referencia",
    ]
].sort_values(
    ["ano_referencia", "mes_referencia", "ano_modelo"]
)

,ano_modelo,zero_km,valor_centavos,ano_referencia,mes_referencia
967,NaN,True,528400,2001,1
966,NaN,True,544200,2001,2
965,NaN,True,560500,2001,3
964,NaN,True,577200,2001,4
313,2000.0,False,544000,2001,5
...,...,...,...,...,...
315,2001.0,False,342000,2026,7
619,2002.0,False,374600,2026,7
10,2000.0,False,320600,2026,8
314,2001.0,False,340900,2026,8


In [217]:
structural_checks = {
    "invalid_month": ~df_history["mes_referencia"].between(1, 12),

    "invalid_model_year": (
        df_history["ano_modelo"].notna()
        & (
            df_history["ano_modelo"]
            > df_history["ano_referencia"] + 1
        )
    ),

    "invalid_price": df_history["valor_centavos"] <= 0,

    "invalid_fipe_code": ~df_history["codigo_fipe"].str.fullmatch(
        r"\d{6}-\d"
    ),

    "invalid_zero_km_rule": (
        (
            df_history["zero_km"]
            & df_history["ano_modelo"].notna()
        )
        |
        (
            ~df_history["zero_km"]
            & df_history["ano_modelo"].isna()
        )
    ),
}

In [218]:
structural_summary = {
    rule: mask.sum()
    for rule, mask in structural_checks.items()
}

structural_summary

{'invalid_month': np.int64(0),
 'invalid_model_year': np.int64(0),
 'invalid_price': np.int64(22),
 'invalid_fipe_code': np.int64(0),
 'invalid_zero_km_rule': np.int64(0)}

In [219]:
brand_conflict_codes = cardinality_by_fipe[
    cardinality_by_fipe["nome_marca_nunique"] > 1
].index

brand_history = (
    df_history[
        df_history["codigo_fipe"].isin(brand_conflict_codes)
    ]
    .assign(
        reference_date=lambda x: pd.to_datetime(
            dict(
                year=x["ano_referencia"],
                month=x["mes_referencia"],
                day=1,
            )
        )
    )
    .groupby(
        ["codigo_fipe", "nome_marca"],
        dropna=False
    )
    .agg(
        occurrences=("nome_marca", "size"),
        first_reference=("reference_date", "min"),
        last_reference=("reference_date", "max"),
    )
    .reset_index()
    .sort_values(
        ["codigo_fipe", "first_reference"]
    )
)

brand_history

,codigo_fipe,nome_marca,occurrences,first_reference,last_reference
1,040001-7,Buggy,2090,2001-01-01,2021-01-01
0,040001-7,Baby,1000,2018-05-01,2026-08-01
3,040002-5,Buggy,4160,2001-01-01,2018-04-01
2,040002-5,Baby,3206,2018-05-01,2026-08-01
4,832001-2,KIMCO,1201,2001-01-01,2021-02-01
5,832001-2,KYMCO,672,2017-05-01,2026-08-01
6,832002-0,KIMCO,1198,2001-01-01,2021-02-01
7,832002-0,KYMCO,672,2017-05-01,2026-08-01
8,832027-6,KIMCO,563,2001-08-01,2021-02-01
9,832027-6,KYMCO,336,2017-05-01,2026-08-01


In [220]:
brand_month_conflicts = (
    df_history[
        df_history["codigo_fipe"].isin(brand_conflict_codes)
    ]
    .groupby(
        [
            "codigo_fipe",
            "ano_referencia",
            "mes_referencia",
        ]
    )["nome_marca"]
    .nunique()
)

brand_month_conflicts = brand_month_conflicts[
    brand_month_conflicts > 1
]

brand_month_conflicts

codigo_fipe  ano_referencia  mes_referencia
040001-7     2021            1                 2
832001-2     2021            1                 2
                             2                 2
832002-0     2021            1                 2
                             2                 2
832027-6     2021            1                 2
                             2                 2
832028-4     2021            1                 2
                             2                 2
872001-0     2021            1                 3
                             2                 3
872002-9     2021            1                 3
                             2                 3
872003-7     2021            1                 3
                             2                 3
872004-5     2021            1                 3
                             2                 3
Name: nome_marca, dtype: int64

In [221]:
brand_month_conflicts.shape

(17,)